# 01 - Exploration du corpus : chargement, prétraitement et statistiques

In [1]:
import sys
from pathlib import Path

_c = Path.cwd()
while _c != _c.parent and not (_c / "data" / "raw").is_dir():
    _c = _c.parent
import os
os.chdir(_c)
sys.path.insert(0, str(_c))

os.environ["USE_OPENAI_EMBEDDINGS"] = "false"
from src.corpus.chunker import chunk_documents
from src.corpus.loader import load_corpus
from src.corpus.preprocessor import preprocess_documents

C:\Users\Johnson Nancy\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_dir = Path("data/raw")
docs = preprocess_documents(load_corpus(raw_dir))
print(f"{len(docs)} documents charges")
for d in docs:
    print(
        f"  - {d.metadata.get('document', d.metadata.get('source', '?'))} : {len(d.page_content)} caracteres"
    )

9 documents charges
  - conditions_generales.md : 8954 caracteres
  - faq.md : 6971 caracteres
  - formation_commerciaux.md : 9649 caracteres
  - guide_souscription.md : 3548 caracteres
  - notice_information.md : 5386 caracteres
  - spec_groupe_partenariats.md : 1657 caracteres
  - spec_produits.md : 6321 caracteres
  - spec_produits_commerciaux.md : 3274 caracteres
  - spec_produits_details.md : 2827 caracteres


In [3]:
from collections import Counter

chunks = chunk_documents(docs)

print(f"{len(chunks)} chunks produits")

print(Counter(c.metadata.get("document", "?") for c in chunks))

241 chunks produits
Counter({'formation_commerciaux.md': 51, 'conditions_generales.md': 41, 'spec_produits.md': 37, 'faq.md': 31, 'notice_information.md': 24, 'spec_produits_commerciaux.md': 18, 'guide_souscription.md': 17, 'spec_produits_details.md': 14, 'spec_groupe_partenariats.md': 8})


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sizes = [(300, 30), (300, 60), (500, 50), (500, 100), (800, 80), (800, 160)]
for size, overlap in sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " "],
    )
    print(size, overlap, "->", len(splitter.split_documents(docs)), "chunks")

300 30 -> 241 chunks
300 60 -> 248 chunks
500 50 -> 149 chunks
500 100 -> 151 chunks
800 80 -> 93 chunks
800 160 -> 93 chunks
